In [3]:
#!/usr/bin/env python3
# ============================================================
# Day 4 — Pipeline Parallel (GPipe-style) Toy Skeleton
# Learning goal:
# - understand Pipeline Parallel splits layers, not tensors
# - understand microbatch flow
# - understand stage-to-stage activation transfer
# - understand bubble concept from timeline/logging
#
# IMPORTANT:
# - Core logic is intentionally left as TODOs
# - Fill them by yourself
# - Start with the simplest possible pipeline
# ============================================================

import time
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.optim as optim


# ============================================================
# Config
# ============================================================

@dataclass
class Config:
    input_dim: int = 128
    hidden_dim: int = 256
    num_classes: int = 10

    batch_size: int = 32
    num_microbatches: int = 4

    lr: float = 1e-3
    num_steps: int = 5

    device0: str = "cuda:0"
    device1: str = "cuda:1"


# ============================================================
# Stage modules
# ============================================================

class Stage0(nn.Module):
    """
    First half of the model.
    Put only the front layers here.
    """
    def __init__(self, input_dim: int, hidden_dim: int):
        super().__init__()

        # TODO:
        # Build the first half of the network.
        #
        # Hint:
        # A simple choice:
        #   Linear(input_dim, hidden_dim)
        #   ReLU
        #   Linear(hidden_dim, hidden_dim)
        #   ReLU
        #
        # self.net = ...
        # raise NotImplementedError("TODO: implement Stage0")
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),   
        )
           

    def forward(self, x):
        # TODO:
        # Return the activation produced by Stage0
        # raise NotImplementedError("TODO: implement Stage0.forward")
        return self.net(x)


class Stage1(nn.Module):
    """
    Second half of the model.
    Put only the back layers here.
    """
    def __init__(self, hidden_dim: int, num_classes: int):
        super().__init__()

        # TODO:
        # Build the second half of the network.
        #
        # Hint:
        # A simple choice:
        #   Linear(hidden_dim, hidden_dim)
        #   ReLU
        #   Linear(hidden_dim, num_classes)
        #
        # self.net = ...
        # raise NotImplementedError("TODO: implement Stage1")
        self.net = nn.Sequential(
        nn.Linear(hidden_dim, hidden_dim),
        nn.ReLU(),
        nn.Linear(hidden_dim, num_classes),
        nn.ReLU(),   
        )
       

    def forward(self, h):
        # TODO:
        # Return the final logits
        # raise NotImplementedError("TODO: implement Stage1.forward")
        return self.net(h)


# ============================================================
# Utilities
# ============================================================

def make_batch(cfg: Config):
    """
    Create a toy classification batch.
    """
    x = torch.randn(cfg.batch_size, cfg.input_dim)
    y = torch.randint(low=0, high=cfg.num_classes, size=(cfg.batch_size,))
    return x, y


def split_microbatches(x: torch.Tensor, y: torch.Tensor, num_microbatches: int):
    """
    Split one batch into multiple microbatches.

    TODO:
    - Split x along batch dimension
    - Split y along batch dimension
    - Return a list of (x_mb, y_mb)

    Hint:
    - torch.chunk(..., dim=0)
    - Ensure x and y are split into the same number of parts
    """
    # raise NotImplementedError("TODO: implement split_microbatches")
    x_chunks = torch.chunk(x, num_microbatches, dim = 0)
    y_chunks = torch.chunk(y, num_microbatches, dim = 0)

    if not(len(x_chunks) == len(y_chunks)):
         raise RuntimeError("x and y were split into different numbers of chunks.")
        
    microbatches = list(zip(x_chunks, y_chunks))
    return microbatches


def print_timeline_header(step: int):
    print("\n" + "=" * 70)
    print(f"TRAIN STEP {step}")
    print("=" * 70)


# ============================================================
# 1) Baseline: no microbatch
# ============================================================

def train_step_no_microbatch(
    cfg: Config,
    stage0: nn.Module,
    stage1: nn.Module,
    criterion,
    optim0,
    optim1,
    x: torch.Tensor,
    y: torch.Tensor,
):
    """
    Simplest baseline:
    whole batch goes through stage0 first, then stage1.

    This is NOT yet a real useful pipeline.
    It is only to help you understand:
    - model split across devices
    - activation transfer between stages
    - end-to-end backward across stages
    """
    stage0.train()
    stage1.train()

    optim0.zero_grad()
    optim1.zero_grad()

    # TODO:
    # Move x to device0 and y to device1 (or move target when needed)
    #
    # Hint:
    # x0 = ...
    # y1 = ...
    # raise NotImplementedError("TODO: move tensors to devices")
    x0 = x.to(cfg.device0)
    y1 = y.to(cfg.device1)

    t0 = time.time()

    print("[No-MB] Stage0 forward starts")
    # TODO:
    # Run Stage0 on device0
    # h = ...
    # raise NotImplementedError("TODO: stage0 forward")
    h = stage0(x0)
    

    print("[No-MB] Transfer activation Stage0 -> Stage1")
    # TODO:
    # Move activation to device1
    # h1 = ...
    # raise NotImplementedError("TODO: activation transfer to device1")
    h1 = h.to(cfg.device1)

    print("[No-MB] Stage1 forward starts")
    # TODO:
    # Run Stage1 on device1
    # logits = ...
    # raise NotImplementedError("TODO: stage1 forward")
    logits = stage1(h1)

    print("[No-MB] Compute loss")
    # TODO:
    # loss = ...
    # raise NotImplementedError("TODO: compute loss")
    loss = criterion(logits, y1)

    print("[No-MB] Backward starts")
    # TODO:
    # loss.backward()
    # raise NotImplementedError("TODO: backward")
    loss.backward()

    print("[No-MB] Optimizer step")
    # TODO:
    # optim0.step()
    # optim1.step()
    # raise NotImplementedError("TODO: optimizer step")
    optim0.step()
    optim1.step()


    t1 = time.time()
    print(f"[No-MB] loss={float(loss):.6f} | step_time={t1 - t0:.4f}s")

    return float(loss)


# ============================================================
# 2) Simple microbatch version
# ============================================================

def train_step_microbatch_simple(
    cfg: Config,
    stage0: nn.Module,
    stage1: nn.Module,
    criterion,
    optim0,
    optim1,
    x: torch.Tensor,
    y: torch.Tensor,
):
    """
    Simple microbatch version.

    IMPORTANT:
    This is still NOT a fully optimized GPipe scheduler.
    It is just the easiest educational version:
    process microbatches one by one in order.

    Goal:
    - understand that one batch is split into smaller chunks
    - understand each microbatch produces an activation
    - understand activations flow from stage0 to stage1
    """
    stage0.train()
    stage1.train()

    optim0.zero_grad()
    optim1.zero_grad()

    # TODO:
    # Split batch into microbatches
    # microbatches = ...
    # raise NotImplementedError("TODO: split into microbatches")
    microbatches = split_microbatches(x, y, cfg.num_microbatches)

    total_loss = 0.0
    t0 = time.time()

    # TODO:
    # Loop over microbatches
    #
    # For each microbatch:
    #   1) move x_mb to device0
    #   2) forward Stage0
    #   3) transfer activation to device1
    #   4) move y_mb to device1
    #   5) forward Stage1
    #   6) compute loss_mb
    #   7) accumulate total_loss
    #
    # IMPORTANT:
    # - Decide whether to divide each microbatch loss by num_microbatches
    #   before backward, so that the full-batch gradient scale stays stable.
    #
    # Suggested logging:
    #   [MB 0] Stage0 forward
    #   [MB 0] Send activation to Stage1
    #   [MB 0] Stage1 forward
    #   [MB 0] Compute loss
    #
    # You may either:
    # A) store losses and backward once on summed loss
    # or
    # B) backward per microbatch immediately
    #
    # For learning, approach B is often easier to reason about.
    # raise NotImplementedError("TODO: microbatch forward/loss logic")
    total_loss = 0.0
    t0 = time.time()
    for i, (x_mb, y_mb) in enumerate(microbatches):
        print(f"[MB {i}] Stage0 forward starts")

        # move current microbatch input to Stage0 device
        x0 = x_mb.to(cfg.device0)

        # Forward on Stage0
        h = stage0(x0)

        # Move activation to Stage1 device
        h1 = h.to(cfg.device1)

        print(f"[MB {i}] Stage1 forward starts")


        # move target to Stage1 device, loss is computed there
        y1 = y_mb.to(cfg.device1)

        #forward on Stage1
        logits = stage1(h1)

        print(f"[MB {i}] Compute loss")

        # Divide by num_microbatches so accumulated gradients are on a comparable scale
        loss_mb = criterion(logits, y1) / cfg.num_microbatches

        # For display only, recover the original per-microbatch loss scale
        total_loss += float(loss_mb.detach()) * cfg.num_microbatches

        print(f"[MB {i}] Backward starts")
        loss_mb.backward()
    
   
    print("[MB] Backward / optimizer step")

    # TODO:
    # If you did not backward inside loop, do it here.
    # Then optimizer step for both stages.
    # raise NotImplementedError("TODO: backward and optimizer step")
    optim0.step()
    optim1.step()

    t1 = time.time()
    print(f"[MB] total_loss={total_loss:.6f} | step_time={t1 - t0:.4f}s")

    return total_loss


# ============================================================
# Main
# ============================================================

def main():
    cfg = Config()

    # Safety check
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for this exercise.")
    if torch.cuda.device_count() < 2:
        raise RuntimeError("This exercise expects at least 2 GPUs.")

    print("Using devices:", cfg.device0, cfg.device1)

    # TODO:
    # Instantiate stages
    # stage0 = ...
    # stage1 = ...
    # raise NotImplementedError("TODO: create stage modules")
    stage0 = Stage0(cfg.input_dim, cfg.hidden_dim)
    stage1 = Stage1(cfg.hidden_dim, cfg.num_classes)

    # TODO:
    # Move stage0 to device0, stage1 to device1
    # raise NotImplementedError("TODO: move stages to GPUs")
    # Place stages on different GPUs
    stage0 = stage0.to(cfg.device0)
    stage1 = stage1.to(cfg.device1)

    # TODO:
    # Define criterion
    # Example: nn.CrossEntropyLoss()
    # raise NotImplementedError("TODO: define loss function")
    criterion = nn.CrossEntropyLoss()

    # TODO:
    # Define separate optimizers for stage0 and stage1
    # raise NotImplementedError("TODO: define optimizers")
    # Separate optimizers for each stage
    optim0 = optim.Adam(stage0.parameters(), lr=cfg.lr)
    optim1 = optim.Adam(stage1.parameters(), lr=cfg.lr)  

    print("\n--- Phase A: no microbatch baseline ---")
    for step in range(2):
        print_timeline_header(step)
        x, y = make_batch(cfg)
        train_step_no_microbatch(
            cfg, stage0, stage1, criterion, optim0, optim1, x, y
        )

    print("\n--- Phase B: microbatch version ---")
    for step in range(cfg.num_steps):
        print_timeline_header(step)
        x, y = make_batch(cfg)
        train_step_microbatch_simple(
            cfg, stage0, stage1, criterion, optim0, optim1, x, y
        )

    print("\nDone.")


if __name__ == "__main__":
    main()

Using devices: cuda:0 cuda:1

--- Phase A: no microbatch baseline ---

TRAIN STEP 0
[No-MB] Stage0 forward starts
[No-MB] Transfer activation Stage0 -> Stage1
[No-MB] Stage1 forward starts
[No-MB] Compute loss
[No-MB] Backward starts
[No-MB] Optimizer step
[No-MB] loss=2.299401 | step_time=0.3275s

TRAIN STEP 1
[No-MB] Stage0 forward starts
[No-MB] Transfer activation Stage0 -> Stage1
[No-MB] Stage1 forward starts
[No-MB] Compute loss
[No-MB] Backward starts
[No-MB] Optimizer step
[No-MB] loss=2.304465 | step_time=0.0030s

--- Phase B: microbatch version ---

TRAIN STEP 0
[MB 0] Stage0 forward starts
[MB 0] Stage1 forward starts
[MB 0] Compute loss
[MB 0] Backward starts
[MB 1] Stage0 forward starts
[MB 1] Stage1 forward starts
[MB 1] Compute loss
[MB 1] Backward starts
[MB 2] Stage0 forward starts
[MB 2] Stage1 forward starts
[MB 2] Compute loss
[MB 2] Backward starts
[MB 3] Stage0 forward starts
[MB 3] Stage1 forward starts
[MB 3] Compute loss
[MB 3] Backward starts
[MB] Backward / o

/tmp/ipykernel_55/495948028.py:246: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(f"[No-MB] loss={float(loss):.6f} | step_time={t1 - t0:.4f}s")
